# Crisis Mining (Colab)

Find positions where the policy fails, solve with deep search.
Phase 1: Policy probes (instant). Phase 2: Parallel GPU replays.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code
2. `pillar2u_epoch_8.pt` — model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar2u_epoch_8.pt /content/alphatrain/data/model.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === EDIT THESE ===
SEED_START = 100000
SEED_END = 101000
WORKERS = 20
BATCH_SIZE = 128
SAVE_DIR = f'{DRIVE}/crisis_v1'
# ==================

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} seeds)')
print(f'Workers: {WORKERS}, Save: {SAVE_DIR}')

!python -m alphatrain.scripts.crisis_mining \
    --model /content/alphatrain/data/model.pt \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --recovery-turns 25 --recovery-sims 2000 \
    --prevention-turns 50 --prevention-sims 1600 \
    --continue-turns 500 \
    --workers {WORKERS} --device cuda --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR}